# 00. Colab 세팅
런타임 → 런타임 유형 변경 → **L4 GPU** 선택 후 위에서부터 실행.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print(torch.__version__, torch.cuda.is_available())

## Drive 마운트 (데이터·체크포인트는 Drive에 보관: 세션이 끊겨도 유지)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/weather_forecasting'
os.makedirs(WORK + '/data/raw', exist_ok=True)
os.makedirs(WORK + '/checkpoints', exist_ok=True)

## 코드 가져오기 (공개 저장소라 인증 불필요)

In [ ]:
%cd /content
!if [ -d weather_forecasting ]; then git -C weather_forecasting pull; else git clone https://github.com/qruse/weather_forecasting.git; fi
%cd /content/weather_forecasting
!pip install -q -r requirements.txt
# gitignore 된 asos/era5/checkpoints 만 Drive 로 연결 (추적 파일은 건드리지 않음)
!mkdir -p data/raw {WORK}/data/raw/asos {WORK}/data/raw/era5
!rm -rf data/raw/asos data/raw/era5
!ln -sfn {WORK}/data/raw/asos data/raw/asos
!ln -sfn {WORK}/data/raw/era5 data/raw/era5
!ln -sfn {WORK}/checkpoints checkpoints

## 인증키 (왼쪽 🔑 Secrets 에 `DATA_GO_KR_KEY` 등록 후 노트북 액세스 허용)

In [ ]:
from google.colab import userdata
try:
    os.environ['DATA_GO_KR_KEY'] = userdata.get('DATA_GO_KR_KEY')
except Exception as e:
    print('키 없음 (ASOS 다운로드는 건너뜀):', e)

## 데이터 (이미 있으면 건너뜀)

In [ ]:
import pathlib
if not pathlib.Path('data/raw/era5/seoul.csv').exists():
    !python scripts/download_era5.py 2018-01-01 2025-12-31
if 'DATA_GO_KR_KEY' in os.environ and not pathlib.Path('data/raw/asos/seoul.csv').exists():
    !python scripts/download_asos.py 2018-01-01 2025-12-31
!ls -la data/raw/*